Loading the sample and the model

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sample = pd.read_pickle("../artifacts/sample.pkl")
model = SentenceTransformer("all-MiniLM-L6-v2")
model.max_seq_length = 128   # ~100 words; covers most reviews and keeps it fast
print(len(sample), "reviews to embed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

50001 reviews to embed


Testing on 5 sentences first

In [2]:
test = [
    "The coffee tasted stale and bitter",
    "These beans were old and had a burnt flavor",
    "My dog loves these treats",
    "The box arrived crushed and leaking",
    "Great price for the quantity",
]
v = model.encode(test, normalize_embeddings=True)
sim = pd.DataFrame(v @ v.T, index=test, columns=range(5)).round(2)
sim

,0,1,2,3,4
The coffee tasted stale and bitter,1.00,0.57,0.14,0.21,0.11
These beans were old and had a burnt flavor,0.57,1.00,0.30,0.18,0.16
My dog loves these treats,0.14,0.30,1.00,0.17,0.17
The box arrived crushed and leaking,0.21,0.18,0.17,1.00,0.18
Great price for the quantity,0.11,0.16,0.17,0.18,1.00


Embedding all 50K reviews

In [3]:
texts = sample["text_for_embedding"].tolist()

t0 = time.time()
emb = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # unit length, so a plain dot product = cosine similarity
    convert_to_numpy=True,
).astype(np.float32)
print(f"Done: {emb.shape} in {time.time() - t0:.0f}s")

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Done: (50001, 384) in 278s


Saving the embeddings

In [4]:
np.save("../artifacts/embeddings.npy", emb)
print("Saved ../artifacts/embeddings.npy")

Saved ../artifacts/embeddings.npy


Trying a first semantic search

In [6]:
def search(query, k=5, df=sample, E=emb):
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = E @ q
    top = np.argsort(-scores)[:k]
    out = df.iloc[top][["Score", "Summary", "Text"]].copy()
    out["similarity"] = scores[top].round(3)
    out["Text"] = out["Text"].str[:200]
    return out

search("my dog got sick")

,Score,Summary,Text,similarity
22046,5,they are not sick anymore,"I have three dog friends. Maggie, my 14 year o...",0.587
43150,2,Dog stopped eating,Our pup stopped eating this food. Took me a fe...,0.581
9939,1,Made my dog sick,"These made my 23-lb, 1 1/2 year old dog sick. ...",0.568
46031,1,Made my dog sick,"I was looking for a quality food for my 90LBS,...",0.558
1839,1,WARNING!!!,"Our 11 week old Bullmastiff puppy, Jolene, die...",0.553
